# Phase 6: Validation Hyperparameter Tuning & Multi-Paradigm Ensemble Blending
**Course: ES 335 - Machine Learning • IIT Gandhinagar**
**Authors:** Devarsh Patel (24110245) & Dhrumil Bhutaiya (24110075)

In Phase 5, the **Non-Linear RBF Kernel SVM** was identified as the champion candidate architecture (82.14% Accuracy, 0.9038 ROC-AUC), while **HistGradientBoosting** proved to be the superior tree-based model (80.66% Accuracy in 3.97s).

Phase 6 conducts rigorous validation-split optimization across two core thrusts:
1. **Systematic 2D Grid Search for RBF Kernel SVM:** Sweeping soft-margin penalty $C \in [0.5, 1.0, 2.0, 3.0, 4.0]$, Gaussian kernel bandwidth $\gamma \in [0.010, \text{'scale'}, 0.025]$, and cost-sensitive diagnostic class weighting.
2. **Multi-Paradigm Ensemble Blending:** Constructing a soft-voting blend fusing the smooth continuous distance scores of RBF Kernel SVM with the discrete orthogonal decision splits of HistGradientBoosting to break the $0.9100$ ROC-AUC barrier.

> **Strict Protocol:** All tuning and grid explorations are conducted strictly on the **5,000 unseen validation patches**. The test set ($N=5,000$) remains completely untouched until Phase 7.

In [1]:
# ==============================================================================
# 1. Environment Setup & Dependency Loading
# ==============================================================================
import os
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
PLOTS_DIR = PROJECT_ROOT / "outputs" / "plots"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"

print("[OK] Phase 6 Tuning Environment initialized.")
print("Loaded tuning grids and trained model artifacts successfully.")

[OK] Phase 6 Tuning Environment initialized.
Loaded tuning grids and trained model artifacts successfully.


## 2. RBF Kernel SVM Hyperparameter Tuning Grid Search Results
We inspect the full 11-configuration tuning grid evaluated on 40,000 training patches and 5,000 validation patches.

In [2]:
df_grid = pd.read_csv(RESULTS_DIR / "rbf_svm_tuning_grid.csv")
disp_cols = ["C", "gamma", "class_weight", "accuracy", "roc_auc", "recall_sensitivity", "precision", "f1_score", "support_vectors", "fit_time_sec"]
fmt_df = df_grid.sort_values(by="roc_auc", ascending=False)[disp_cols].copy()
fmt_df["accuracy"] = (fmt_df["accuracy"] * 100).round(2).astype(str) + "%"
fmt_df["recall_sensitivity"] = (fmt_df["recall_sensitivity"] * 100).round(2).astype(str) + "%"
fmt_df["precision"] = (fmt_df["precision"] * 100).round(2).astype(str) + "%"
fmt_df["roc_auc"] = fmt_df["roc_auc"].round(4)
fmt_df["f1_score"] = fmt_df["f1_score"].round(4)

print("=" * 105)
print("RBF KERNEL SVM HYPERPARAMETER TUNING GRID (Validation Set: 5,000 Patches):")
print("=" * 105)
print(fmt_df.to_string(index=False))
print("=" * 105)

RBF KERNEL SVM HYPERPARAMETER TUNING GRID (Validation Set: 5,000 Patches):
  C  gamma     class_weight accuracy  roc_auc recall_sensitivity precision  f1_score  support_vectors  fit_time_sec
3.0  scale             None   82.36%   0.9082             78.56%    85.02%    0.8166            15569         94.63
4.0  scale             None   82.38%   0.9080             78.16%    85.36%    0.8160            15315         94.10
2.0  scale {0: 1.0, 1: 1.1}   82.54%   0.9074             80.32%    84.05%    0.8214            15811         92.44
2.0  scale         balanced    82.4%   0.9073             78.88%    84.85%    0.8176            15895         81.89
2.0  scale             None    82.4%   0.9073             78.88%    84.85%    0.8176            15895         89.26
2.0 0.0250             None   82.08%   0.9051             78.04%    84.90%    0.8133            16425        101.70
2.0 0.0100             None   82.16%   0.9048             79.56%    83.92%    0.8168            16128         85.

## 3. 2D Hyperparameter Response Heatmap
We inspect the 2D landscape of Accuracy and ROC-AUC across $C$ and $\gamma$ to visualize optimization stability.

In [3]:
heatmap_path = PLOTS_DIR / "rbf_svm_tuning_heatmap.png"
display(Image(filename=str(heatmap_path), width=950))

## 4. Multi-Paradigm Ensemble Blending (RBF SVM + HistGradientBoosting)
By fusing continuous geometric margins (RBF Kernel SVM) with orthogonal tree splits (HistGradientBoosting), we evaluate soft-voting ensemble blends across weights $w_{\text{RBF}} \in [0.0, 1.0]$.

In [4]:
df_blend = pd.read_csv(RESULTS_DIR / "ensemble_blend_benchmark.csv")
disp_blend = ["weight_rbf", "weight_hgbm", "accuracy", "roc_auc", "rank_auc", "recall_sensitivity", "precision", "f1_score"]
fmt_blend = df_blend[disp_blend].copy()
fmt_blend["accuracy"] = (fmt_blend["accuracy"] * 100).round(2).astype(str) + "%"
fmt_blend["recall_sensitivity"] = (fmt_blend["recall_sensitivity"] * 100).round(2).astype(str) + "%"
fmt_blend["precision"] = (fmt_blend["precision"] * 100).round(2).astype(str) + "%"
fmt_blend["roc_auc"] = fmt_blend["roc_auc"].round(4)
fmt_blend["rank_auc"] = fmt_blend["rank_auc"].round(4)
fmt_blend["f1_score"] = fmt_blend["f1_score"].round(4)

print("=" * 100)
print("ENSEMBLE BLEND BENCHMARK ACROSS BLEND WEIGHTS (Validation Set: 5,000 Patches):")
print("=" * 100)
print(fmt_blend.to_string(index=False))
print("=" * 100)
print("Optimal Blend Configuration: 70% RBF Kernel SVM + 30% HistGradientBoosting")
print("  -> ROC-AUC reaches project-record: 0.9108 (Rank-AUC = 0.9114)")

ENSEMBLE BLEND BENCHMARK ACROSS BLEND WEIGHTS (Validation Set: 5,000 Patches):
 weight_rbf  weight_hgbm accuracy  roc_auc  rank_auc recall_sensitivity precision  f1_score
       0.00         1.00   80.66%   0.8990    0.8990             74.44%    85.02%    0.7938
       0.10         0.90   81.08%   0.9018    0.9025             74.96%    85.41%    0.7985
       0.20         0.80   81.24%   0.9042    0.9054             75.56%    85.24%    0.8011
       0.30         0.70   81.46%   0.9062    0.9078             75.68%    85.57%    0.8032
       0.40         0.60   81.72%   0.9080    0.9096             76.04%    85.79%    0.8062
       0.50         0.50   81.92%   0.9093    0.9109             76.76%    85.59%    0.8094
       0.60         0.40   81.90%   0.9103    0.9114             76.72%    85.59%    0.8091
       0.70         0.30   82.16%   0.9108    0.9114             77.32%    85.61%    0.8125
       0.80         0.20   82.26%   0.9106    0.9109             77.76%    85.45%    0.8142
 

## 5. Visualizing the Blended ROC Curve vs. Single Architectures

In [5]:
roc_blend_path = PLOTS_DIR / "ensemble_vs_single_roc.png"
display(Image(filename=str(roc_blend_path), width=850))

## 6. Key Strategic Takeaways from Phase 6

1. **RBF SVM Tuning Delivers Clear Gains:**
   - Increasing $C$ from $1.0 \to 3.0$ with $\gamma = \text{'scale'}$ lifted ROC-AUC from $0.9038 \to \mathbf{0.9082}$ and Precision to $\mathbf{85.02\%}$.
   - Cost-sensitive weighting ($C=2.0, \text{weight}=\{0: 1.0, 1: 1.1\}$) pushed Sensitivity past $80\%$ ($80.32\%$) with project-record **82.54% Accuracy** and **0.8214 F1-score**.
2. **Multi-Paradigm Ensembling Breaks the 0.9100 Barrier:**
   - The soft-voting blend of $70\%$ RBF Kernel SVM $+ 30\%$ HistGradientBoosting achieved **0.9108 ROC-AUC** and **0.9114 Rank-AUC**, outperforming both individual models.
   - The diversity between smooth Gaussian distance boundaries and orthogonal decision trees eliminates individual edge-case errors.
3. **Freeze Decision for Phase 7 (One-Shot Test Evaluation):**
   - We freeze two optimal candidates for the final test set:
     - **Primary Single Model Champion:** Tuned RBF Kernel SVM ($C=2.0, \gamma=\text{'scale'}, \text{weight}=\{0: 1.0, 1: 1.1\}$) — 82.54% Acc, 80.32% Sens, 0.9074 AUC.
     - **Primary Ensemble Champion:** 70/30 Soft-Voting Blend (RBF SVM + HistGBM) — 82.16% Acc, 0.9108 ROC-AUC.